<a href="https://colab.research.google.com/github/bforsbe/ChimeraX/blob/develop/Fold_znf_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Evolving protein sequences using ESMFold/OmegaFold**

In a previous session ago you mutated a DNA sequence to get a high mutation score, and tried to keep the fold intact. Today, we will use the same tool to probe many sequences to learn something about sequence space and and fold-space.

###Definitions
> **Sequence space**: The set of all possible sequences
> **Fold-space**: The set of all possible sequences that fold into a stable, defined shape

First, we will generate completely random mutations of a base sequence and look at how similar they are to the base structure. Then, we will exert evolutionary pressure to make sure that sequences are part of folded space, and compare.

**Why?** Because this is a model of evolution, in the sense that there is a "force" keeping DNA and protein sequences functional. Today, that force is similarity to the original structure, whereas in nature it is probably keeping function. This is not quite the same, but since fold and function are so closely connected, it is a useful enough proxy.

<details><summary>More about the software</summary>

ESMfold is a protein language model that has essentially learnt a mapping from sequence to structure, without needing anything else. Itäs not the best or most accurate tool, but we will pretend it is, because its decent and runs quickly enough to learn something importnat. For more details on ESMfold see: [Github](https://github.com/facebookresearch/esm/tree/main/esm), [Preprint](https://www.biorxiv.org/content/10.1101/2022.07.20.500902v1)

Omegafold is another protein language model, : [Github](https://github.com/HeliXonProtein/OmegaFold), [Preprint](https://www.biorxiv.org/content/10.1101/2022.07.21.500999v1)

</details>

## Zinc finger
We will work with another sequence today, which is a zinc-finger domain. This is a small protein domain that is usually part of larger proteins, but which does fold on its own. Nature has reinvented this domain many times, which will make the sequence analysis more interesting.

In [ ]:
%%time

#@title  install ESMFold, OpenFold and download Params (~4min)
version = "1"
model_name = "esmfold_v0.model" if version == "0" else "esmfold.model"
import os,sys,re,time
if not os.path.isfile(model_name):
  # download esmfold params
  os.system("apt-get install aria2 -qq")
  os.system(f"aria2c -q -x 16 https://colabfold.steineggerlab.workers.dev/esm/{model_name} &")

  if not os.path.isfile("finished_install"):
    # install libs
    print("installing libs...")
    os.system("pip install -q omegaconf pytorch_lightning biopython ml_collections einops py3Dmol modelcif")
    os.system("pip install -q git+https://github.com/NVIDIA/dllogger.git")

    print("installing openfold...")
    # install openfold
    os.system(f"pip install -q git+https://github.com/sokrypton/openfold.git")

    print("installing esmfold...")
    # install esmfold
    os.system(f"pip install -q git+https://github.com/sokrypton/esm.git")
    os.system("touch finished_install")

  # wait for Params to finish downloading...
  while not os.path.isfile(model_name):
    time.sleep(5)
  if os.path.isfile(f"{model_name}.aria2"):
    print("downloading params...")
  while os.path.isfile(f"{model_name}.aria2"):
    time.sleep(5)

If ESMfold does not work (it has happend occasionally in the past, you can switch to OmegaFold, which is much slower as tested here). **You do *NOT* need to install it if using ESMfold**.

In [ ]:
#@markdown ##Install OmegaFold
import os,sys,re
from IPython.utils import io
if "SETUP_DONE" not in dir():
  import torch
  device = "cuda" if torch.cuda.is_available() else "cpu"
  with io.capture_output() as captured:
    if not os.path.isdir("OmegaFold"):
      %shell git clone --branch beta --quiet https://github.com/sokrypton/OmegaFold.git
      # %shell cd OmegaFold; pip -q install -r requirements.txt
      %shell pip -q install py3Dmol biopython==1.81
      %shell apt-get install aria2 -qq > /dev/null
      %shell aria2c -q -x 16 https://helixon.s3.amazonaws.com/release1.pt
      %shell mkdir -p ~/.cache/omegafold_ckpt
      %shell mv release1.pt ~/.cache/omegafold_ckpt/model.pt
  SETUP_DONE = True

## Helper Functions

This section defines all reusable functions needed throughout the notebook for tasks like sequence hashing, PDB renumbering, RMSD calculation, sequence comparison, translation, and scoring. These functions are loaded once to streamline the rest of the notebook.

In [ ]:
# @title
import os, re, requests
import hashlib
import torch
import numpy as np
import tempfile
from IPython.display import display, HTML
from textwrap import wrap
import Bio
from Bio.Seq import Seq
from Bio.PDB import PDBParser, Superimposer, PDBIO

# --- Global variables and constants ---
alphabet_list = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')

# Define device here to ensure it's available (copied from OmegaFold install)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Genetic code (DNA codons) (copied from z3HeD2V8epVW)
codon_table = {
    "TTT":"F","TTC":"F","TTA":"L","TTG":"L",
    "CTT":"L","CTC":"L","CTA":"L","CTG":"L",
    "ATT":"I","ATC":"I","ATA":"I","ATG":"M",
    "GTT":"V","GTC":"V","GTA":"V","GTG":"V",
    "TCT":"S","TCC":"S","TCA":"S","TCG":"S",
    "CCT":"P","CCC":"P","CCA":"P","CCG":"P",
    "ACT":"T","ACC":"T","ACA":"T","ACG":"T",
    "GCT":"A","GCC":"A","GCA":"A","GCG":"A",
    "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*",
    "CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
    "AAT":"N","AAC":"N","AAA":"K","AAG":"K",
    "GAT":"D","GAC":"D","GAA":"E","GAG":"E",
    "TGT":"C","TGC":"C","TGA":"*","TGG":"W",
    "CGT":"R","CGC":"R","CGA":"R","CGG":"R",
    "AGT":"S","AGC":"S","AGA":"R","AGG":"R",
    "GGT":"G","GGC":"G","GGA":"G","GGG":"G"
}

reverse_codon_table = {}

for codon, amino_acid in codon_table.items():
    reverse_codon_table[amino_acid] = codon

# Original DNA sequence (taken from z3HeD2V8epVW and made global)
_original_dna_seq_global_str = "tacaaatgcgggttgtgcgagaggagctttgtggaaaagtctgccctttccagacaccagagggttcacaaaaat".replace(" ", "").upper()

# --- Helper Function Definitions ---

def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()

def renum_pdb_str(pdb_str, Ls=None, renum=True, offset=1):
  # Copied from CFCwEAa2oZEN
  if Ls is not None:
    L_init = 0
    new_chain = {}
    for L,c in zip(Ls, alphabet_list):
      new_chain.update({i:c for i in range(L_init,L_init+L)})
      L_init += L

  n,num,pdb_out = 0,offset,[]
  resnum_ = None
  chain_ = None
  new_chain_ = new_chain[0] if Ls else None
  for line in pdb_str.split("\n"):
    if line[:4] == "ATOM":
      chain = line[21:22]
      resnum = int(line[22:22+5])
      if resnum_ is None: resnum_ = resnum
      if chain_ is None: chain_ = chain
      if resnum != resnum_ or chain != chain_:
        num += (resnum - resnum_)
        n += 1
        resnum_,chain_ = resnum,chain
      if Ls is not None:
        if new_chain[n] != new_chain_:
          num = offset
          new_chain_ = new_chain[n]
      N = num if renum else resnum
      if Ls is None: pdb_out.append("%s%4i%s" % (line[:22],N,line[26:]))
      else: pdb_out.append("%s%s%4i%s" % (line[:21],new_chain[n],N,line[26:]))
  return "\n".join(pdb_out)

def align_pred_to_ref(pred_pdb_str, ref_pdb_file, out_file="pred_aligned.pdb"):
    # Copied from c51BDdsXe68l
    parser = PDBParser(QUIET=True)

    # Save predicted PDB string to temp file
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdb") as tmp_pred:
        tmp_pred.write(pred_pdb_str.encode())
        pred_file = tmp_pred.name

    pred = parser.get_structure("pred", pred_file)
    ref  = parser.get_structure("ref", ref_pdb_file)

    # Use CA atoms for alignment
    pred_atoms = [a for a in pred.get_atoms() if a.get_id() == "CA"]
    ref_atoms  = [a for a in ref.get_atoms()  if a.get_id() == "CA"]

    n = min(len(pred_atoms), len(ref_atoms))
    si = Superimposer()
    si.set_atoms(ref_atoms[:n], pred_atoms[:n])  # align prediction to reference
    si.apply(pred.get_atoms())                   # apply transform to prediction

    # Write aligned prediction
    io = PDBIO()
    io.set_structure(pred)
    io.save(out_file)
    os.remove(pred_file) # Clean up temporary file

    return out_file, si.rms

def rmsd_to_similarity(rmsd, sigma=2.0):
    # Copied from c51BDdsXe68l
    return np.exp(-rmsd / sigma)

def translate_dna(seq):
    # Copied from z3HeD2V8epVW
    codons = wrap(seq, 3)
    aa = [codon_table.get(c,"?") for c in codons]
    return codons, aa

def compare_sequences(seq1, seq2, space_every=10, spacer=" ", no_number=False):
    # Copied from z3HeD2V8epVW
    if len(seq1) != len(seq2):
        raise ValueError("Sequences must be the same length")

    def highlight_block(s1, s2):
        out1, out2 = [], []
        for a, b in zip(s1, s2):
            out1.append(a)
            if a == b:
                out2.append(b)
            else:
                out2.append(f"<span style='color:red;font-weight:bold'>{b}</span>")
        return "".join(out1), "".join(out2)

    blocks1 = [seq1[i:i+space_every] for i in range(0, len(seq1), space_every)]
    blocks2 = [seq2[i:i+space_every] for i in range(0, len(seq2), space_every)]

    seq1_blocks, seq2_blocks, num_blocks = [], [], []
    pos = 0
    for b1, b2 in zip(blocks1, blocks2):
        o1, o2 = highlight_block(b1, b2)
        seq1_blocks.append(o1)
        seq2_blocks.append(o2)
        pos += len(b1)
        num_blocks.append(str(pos).rjust(len(b1)))

    number_line = spacer.join(num_blocks)
    seq1_line   = spacer.join(seq1_blocks)
    seq2_line   = spacer.join(seq2_blocks)

    html = []
    if no_number:
        html = seq1_line + "<br>" + seq2_line
    else:
        html = number_line + "<br>" + seq1_line + "<br>" + seq2_line
    display(HTML(f"<pre style='font-family:monospace'>{html}</pre>"))

def dna_identity(seq1: str, seq2: str) -> float:
    # Copied from z3HeD2V8epVW
    if len(seq1) != len(seq2):
        raise ValueError("Sequences must have the same length")
    matches = sum(a == b for a, b in zip(seq1.upper(), seq2.upper()))
    return 100 * matches / len(seq1)

def protein_identity(seq1: str, seq2: str) -> float:
    # Copied from z3HeD2V8epVW
    if len(seq1) % 3 != 0 or len(seq2) % 3 != 0:
        raise ValueError("DNA sequences must be divisible by 3")
    prot1 = str(Seq(seq1).translate(to_stop=False))
    prot2 = str(Seq(seq2).translate(to_stop=False))
    if len(prot1) != len(prot2):
        raise ValueError("Translated proteins have different lengths")
    matches = sum(a == b for a, b in zip(prot1, prot2))
    return 100 * matches / len(prot1)

def mutation_score(seq1: str, seq2: str) -> float:
    # Copied from z3HeD2V8epVW
    if len(seq1) != len(seq2):
        raise ValueError("Sequences must have the same length")

    protein_id = protein_identity(seq1, seq2)
    dna_id = dna_identity(seq1, seq2)

    return ((100.-protein_id)+(100.-dna_id))/2

# --- New functions for direct protein-protein comparison (from bbd9326e) ---
def protein_identity_direct(seq1: str, seq2: str) -> float:
    """Return % identity between two protein sequences."""
    if len(seq1) != len(seq2):
        min_len = min(len(seq1), len(seq2))
        # print(f"Warning: Protein sequences have different lengths ({len(seq1)} vs {len(seq2)}). Comparing over shortest length ({min_len}).")
        matches = sum(a == b for a, b in zip(seq1[:min_len], seq2[:min_len]))
        return (100 * matches / min_len) if min_len > 0 else 0.0
    matches = sum(a == b for a, b in zip(seq1.upper(), seq2.upper()))
    return (100 * matches / len(seq1)) if len(seq1) > 0 else 0.0

def protein_mutation_score_direct(seq1: str, seq2: str) -> float:
    """Calculates a simple mutation score based on protein identity."""
    return 100.0 - protein_identity_direct(seq1, seq2)

# --- Download reference PDB (copied from c51BDdsXe68l) ---
ref_file="reference.pdb"
if not os.path.isfile(ref_file):
  print(f"Downloading reference PDB: {ref_file}")
  !wget -q -O {ref_file} https://raw.githubusercontent.com/bforsbe/SK2534/main/reference_omega_znf.pdb

print("Helper functions and global variables loaded.")

## Unconstrained mutation library

The following code will generate mutations at a given sequence identity that you choose. These will then be folded by ESMfold, and their RMSD to the original be saved, and plotted. We will call these **unconstrained**, because it is simply random mutations, regardless of how many.

> **METAPHOR**: Think of our starting sequence as a point in sequence space. If this feels too abstract, imagine it like a single grain of sand in a desert. In that space, a mutated sequence is another point (or grain of sand), and th more mutations it has, the further away it is. This distance is what a score like sequence identity measures.

We call the mutations generated here **unconstrained** because we are simply choosing points in sequence space at a given distance from the starting point, without demanding that they be special in any way.

As before, you can run the cell and click generate, and adjust settings.

In [ ]:
# @title Generate sequences by unconstrained mutation of the base sequences
import random
import ipywidgets as widgets
from IPython.display import display, clear_output

# Amino acids
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'

random.seed(42) # Keep for reproducibility

# --- Interactive Widgets for Sequence Generation ---
original_protein_sequence_text = widgets.Text(
    value='YKCGLCERSFVEKSALSRHQRVHKN', # Default protein sequence from e1026ad4
    description='Original Protein Seq:',
    disabled=False, # Make it editable
    continuous_update=False,
    layout=widgets.Layout(width='600px')
)

seq_id_slider = widgets.FloatSlider(
    value=90,  # Default mutation probability (e.g., 5% of amino acids)
    min=0,
    max=100,
    step=1,
    description='Desired sequence ID (%):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.0f',
)

num_mutants_to_generate_slider = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description='Num. Mutants:',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
)

generate_protein_mutants_button = widgets.Button(
    description='Generate Protein Mutants',
    button_style='success',
    tooltip='Click to generate new protein sequences',
    icon='refresh'
)

output_sequence_area = widgets.Output()

# Global list to store generated protein sequences
global generated_unconstrained_protein_sequences
generated_unconstrained_protein_sequences = []

def generate_and_display_sequences(b=None):
    with output_sequence_area:
        clear_output(wait=True)
        global generated_unconstrained_protein_sequences
        generated_unconstrained_protein_sequences = [] # Clear previous generation

        base_protein_sequence = original_protein_sequence_text.value.upper().strip()
        if not base_protein_sequence:
            print("Please provide a non-empty original protein sequence.")
            return

        random.seed(base_protein_sequence) # Reset seed for deterministic generation per base sequence
        base_protein_len = len(base_protein_sequence)
        mutation_rate = 1 - seq_id_slider.value / 100
        num_to_generate = num_mutants_to_generate_slider.value

        print(f"Original Protein: {base_protein_sequence}")
        print("-" * (len(base_protein_sequence) + 15))

        # Add original sequence to the list, but not count towards num_to_generate if generating new ones
        # This ensures the original is always considered among the pool if needed later.
        # For display purpose, we generate 'num_to_generate' NEW mutants.
        num_amino_acids_to_mutate = int(base_protein_len * mutation_rate)
        if num_amino_acids_to_mutate == 0 and mutation_rate > 0:
            num_amino_acids_to_mutate = 1 # Ensure at least one mutation if rate > 0

        while len(generated_unconstrained_protein_sequences) < num_to_generate: # Generate num_to_generate new sequences
            mutated_protein_list = list(base_protein_sequence)

            positions_to_mutate = np.random.choice(np.arange(base_protein_len), size=num_amino_acids_to_mutate, replace=False)
            for i in np.arange(num_amino_acids_to_mutate):
                mutation_idx = positions_to_mutate[i]
                current_aa = mutated_protein_list[mutation_idx]
                possible_aas = [aa for aa in AMINO_ACIDS if aa != current_aa]
                if possible_aas:
                    mutated_protein_list[mutation_idx] = random.choice(possible_aas)

            mutated_protein_sequence = "".join(mutated_protein_list)

            if mutated_protein_sequence not in generated_unconstrained_protein_sequences:
                generated_unconstrained_protein_sequences.append(mutated_protein_sequence)
                print(f"Generated: {mutated_protein_sequence}")

        print(f"\nGenerated {len(generated_unconstrained_protein_sequences)} unconstrained protein sequences.")
        # print(generated_unconstrained_protein_sequences) # For debug

generate_protein_mutants_button.on_click(generate_and_display_sequences)

print("Configure your original protein sequence, protein mutation probability, and number of mutants, then click 'Generate Protein Mutants':")
display(original_protein_sequence_text,
        seq_id_slider,
        num_mutants_to_generate_slider,
        generate_protein_mutants_button,
        output_sequence_area)

# Initial generation of sequences on load
generate_and_display_sequences()

The currently generated and printed list of sequences are the ones that will get folded (or rather have their fold predicted) by the protein language model below. The sequence and RMSD will get saved, and if you generate new or more sequences, those that it has not seen before will get folded and saved to the same file, so you can keep adding entries.

> **METAPHOR**: In the analogy of a desert being sequence space, and each grain a sequence, calculating the RMSD is asking what color the grain of sand is. The starting sequence is bringht red, and now we are asking how red other grins of sand are. The RMSD is **not** measuring how far away the grains are in sequence space.  

In [ ]:
# @title Fold the generated sequences
import os
import re
import pickle
import numpy as np
import torch
import hashlib
from string import ascii_uppercase, ascii_lowercase
from IPython.display import display, HTML, clear_output
from IPython.utils import io

use_omega = False #@param  {type:"boolean"}
# --- Copy essential helper functions from other cells for self-containment ---
# From CFCwEAa2oZEN
def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()
alphabet_list = list(ascii_uppercase+ascii_lowercase)

def renum_pdb_str(pdb_str, Ls=None, renum=True, offset=1):
  if Ls is not None:
    L_init = 0
    new_chain = {}
    for L,c in zip(Ls, alphabet_list):
      new_chain.update({i:c for i in range(L_init,L_init+L)})
      L_init += L

  n,num,pdb_out = 0,offset,[]
  resnum_ = None
  chain_ = None
  new_chain_ = new_chain[0] if Ls else None # Handle case where Ls might be None
  for line in pdb_str.split("\n"):
    if line[:4] == "ATOM":
      chain = line[21:22]
      resnum = int(line[22:22+5])
      if resnum_ is None: resnum_ = resnum
      if chain_ is None: chain_ = chain
      if resnum != resnum_ or chain != chain_:
        num += (resnum - resnum_)
        n += 1
        resnum_,chain_ = resnum,chain
      if Ls is not None:
        if new_chain[n] != new_chain_:
          num = offset
          new_chain_ = new_chain[n]
      N = num if renum else resnum
      if Ls is None: pdb_out.append("%s%4i%s" % (line[:22],N,line[26:]))
      else: pdb_out.append("%s%s%4i%s" % (line[:21],new_chain[n],N,line[26:]))
  return "\n".join(pdb_out)

# From c51BDdsXe68l
from Bio.PDB import PDBParser, Superimposer, PDBIO
import tempfile

def align_pred_to_ref(pred_pdb_str, ref_pdb_file, out_file="pred_aligned.pdb"):
    parser = PDBParser(QUIET=True)
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdb") as tmp_pred:
        tmp_pred.write(pred_pdb_str.encode())
        pred_file = tmp_pred.name

    pred = parser.get_structure("pred", pred_file)
    ref  = parser.get_structure("ref", ref_pdb_file)

    pred_atoms = [a for a in pred.get_atoms() if a.get_id() == "CA"]
    ref_atoms  = [a for a in ref.get_atoms()  if a.get_id() == "CA"]

    n = min(len(pred_atoms), len(ref_atoms))
    si = Superimposer()
    si.set_atoms(ref_atoms[:n], pred_atoms[:n])
    si.apply(pred.get_atoms())

    io = PDBIO()
    io.set_structure(pred)
    io.save(out_file)
    os.remove(pred_file) # Clean up temporary file

    return out_file, si.rms

def rmsd_to_similarity(rmsd, sigma=2.0):
    return np.exp(-rmsd / sigma)

# --- New functions for direct protein-protein comparison ---
def protein_identity_direct(seq1: str, seq2: str) -> float:
    """Return % identity between two protein sequences."""
    if len(seq1) != len(seq2):
        # For unconstrained generation, lengths might differ, handle this case gracefully
        min_len = min(len(seq1), len(seq2))
        print(f"Warning: Protein sequences have different lengths ({len(seq1)} vs {len(seq2)}). Comparing over shortest length ({min_len}).")
        matches = sum(a == b for a, b in zip(seq1[:min_len], seq2[:min_len]))
        return (100 * matches / min_len) if min_len > 0 else 0.0
    matches = sum(a == b for a, b in zip(seq1.upper(), seq2.upper()))
    return (100 * matches / len(seq1)) if len(seq1) > 0 else 0.0

def protein_mutation_score_direct(seq1: str, seq2: str) -> float:
    """Calculates a simple mutation score based on protein identity."""
    return 100.0 - protein_identity_direct(seq1, seq2)


# --- Configuration ---
unconstrained_history_file = "unconstrained_protein_history.pkl"
reference_pdb_path = "reference.pdb"

# OmegaFold configuration (from CFCwEAa2oZEN)
subbatch_size = 1000
num_cycle = 2
offset_rope = False
device = "cuda" if torch.cuda.is_available() else "cpu"

# Get the original protein sequence from the widget in cell 7c66c75a
# It's assumed this cell (7c66c75a) has been run and its widgets are accessible
try:
    # Accessing widget value directly if original_protein_sequence_text is in global scope
    original_protein_sequence = original_protein_sequence_text.value.upper().strip()
except NameError:
    print("Error: 'original_protein_sequence_text' widget not found. Please ensure cell 7c66c75a is run.")
    original_protein_sequence = ""

if not original_protein_sequence:
    print("No original protein sequence found. Please define it in cell 7c66c75a and run it.")
    # Fallback to a default if absolutely necessary, but user should define
    original_protein_sequence = "YKCGLCERSFVEKSALSRHQRVHKN" # Using the default from 7c66c75a

# Get the list of generated protein sequences from the global variable
try:
    new_generated_sequences = generated_unconstrained_protein_sequences
except NameError:
    print("Error: 'generated_unconstrained_protein_sequences' not found. Please ensure cell 7c66c75a is run.")
    new_generated_sequences = []

# --- Load or Initialize History ---
history = {}
if os.path.exists(unconstrained_history_file):
    with open(unconstrained_history_file, 'rb') as f:
        history = pickle.load(f)
    print(f"Loaded existing history from {unconstrained_history_file} with {len(history)} entries.")
else:
    print(f"No existing history found. Creating new history file: {unconstrained_history_file}")

newly_processed_count = 0

# Ensure reference.pdb exists, if not, download it (copied from c51BDdsXe68l)
if not os.path.isfile(reference_pdb_path):
  print(f"Downloading reference PDB: {reference_pdb_path}")
  !wget -q -O {reference_pdb_path} https://raw.githubusercontent.com/bforsbe/SK2534/main/reference_omega_znf.pdb

# --- Process New Sequences ---
for i, current_protein_sequence in enumerate(new_generated_sequences):
    if current_protein_sequence in history:
        # print(f"Sequence {current_protein_sequence} already in history. Skipping.")
        continue

    print(f"Processing new sequence ({i+1}/{len(new_generated_sequences)}): {current_protein_sequence}")

    try:
        # --- Folding ---
        jobname = f"unconstrained_{get_hash(current_protein_sequence)[:8]}"
        ID = jobname
        seqs = current_protein_sequence.split(":")
        lengths = [len(s) for s in seqs]

        with open(f"{ID}.fasta", "w") as out:
            out.write(f">{ID}\n{current_protein_sequence}\n")

        shell_cmd = f"python OmegaFold/main.py --device={device} --subbatch_size={subbatch_size} --num_cycle={num_cycle} {ID}.fasta ."
        if offset_rope: # Check if offset_rope should be included
            shell_cmd += " --offset_rope"



        if use_omega:
            # Execute OmegaFold prediction
            # Capture output to keep the notebook clean
            with io.capture_output() as captured: # Capture OmegaFold output
                %shell {shell_cmd}
            # Define the predicted PDB file name
            predicted_pdb_file = f"{ID}.pdb"
        else:
            # Execute ESMFold prediction
            if "model" not in dir() or model_name != model_name_:
              if "model" in dir():
                # delete old model from memory
                del model
                gc.collect()
                if torch.cuda.is_available():
                  torch.cuda.empty_cache()

              model = torch.load(model_name, weights_only=False)
              model.eval().cuda().requires_grad_(False)
              model_name_ = model_name

            # optimized for Tesla T4
            model.set_chunk_size(128)

            torch.cuda.empty_cache()
            with io.capture_output() as captured:
                sequence = "".join(open(f"{ID}.fasta").read().splitlines()[1:])

                output = model.infer(sequence, num_recycles=3)

                pdb_str = model.output_to_pdb(output)[0]

                predicted_pdb_file = f"{ID}.pdb"
                with open(predicted_pdb_file, "w") as f:
                    f.write(pdb_str)

        # Read the predicted PDB file
        if not os.path.exists(predicted_pdb_file):
            raise FileNotFoundError(f"OmegaFold did not produce a PDB file for {current_protein_sequence}")

        with open(predicted_pdb_file, 'r') as f:
            predicted_pdb_str = f.read()

        renumbered_pdb_str = renum_pdb_str(predicted_pdb_str, Ls=lengths)

        # --- Scoring ---
        aligned_file, rmsd_val = align_pred_to_ref(renumbered_pdb_str, reference_pdb_path, out_file=f"{ID}_aligned.pdb")
        pred_sim_score = rmsd_to_similarity(rmsd_val)

        seq_id = protein_identity_direct(original_protein_sequence, current_protein_sequence)
        mut_score = protein_mutation_score_direct(original_protein_sequence, current_protein_sequence)
        total_score = mut_score * pred_sim_score

        # --- Store Results ---
        history[current_protein_sequence] = {
            'rmsd': rmsd_val,
            'pred_sim_score': pred_sim_score,
            'seq_id': seq_id, # Protein identity vs original
            'protein_mutation_score': mut_score,
            'total_score': total_score,
            'original_protein_sequence': original_protein_sequence # Store original for context
            # 'pdb_str': renumbered_pdb_str # Optionally store PDB string if needed later
        }
        newly_processed_count += 1
        print(f"  Added: SeqID={seq_id:.2f}%, RMSD={rmsd_val:.2f}Å, Total Score={total_score:.2f}")

        # Clean up temporary OmegaFold files
        os.remove(f"{ID}.fasta")
        os.remove(predicted_pdb_file)
        if os.path.exists(aligned_file): os.remove(aligned_file)

    except Exception as e:
        print(f"  Error processing sequence {current_protein_sequence}: {e}. Skipping this sequence.")


# --- Save Updated History ---
with open(unconstrained_history_file, 'wb') as f:
    pickle.dump(history, f)
print(f"\nSuccessfully processed {newly_processed_count} new sequences.")
print(f"Total unique sequences in history: {len(history)} (saved to {unconstrained_history_file})")


### Comparison between Unconstrained and Constrained Mutations

Lets now visualize the unconstrained mutations (where mutations were generated randomly). The plot below shows the relationship between sequence identity and RMSD, as well as the fraction of sequences that fold correctly (based on an RMSD threshold) across different ranges of sequence identity.

In [ ]:
#@title Plot unconstrained samples
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import pickle
import os
import ipywidgets as widgets # Import ipywidgets
from IPython.display import display # Import display

unconstrained_history_file = "unconstrained_protein_history.pkl"

# Load history (this part remains outside the interactive function as it's static data)
history = {}
if os.path.exists(unconstrained_history_file):
    with open(unconstrained_history_file, 'rb') as f:
        history = pickle.load(f)
    print(f"Loaded history from {unconstrained_history_file} with {len(history)} entries.")
else:
    print(f"No history file found at {unconstrained_history_file}.")

# Prepare data for plotting (this also remains outside)
plot_data = []
for seq, data in history.items():
    plot_data.append({
        'sequence_id': data['seq_id'],
        'rmsd': data['rmsd'],
        'total_score': data['total_score'],
        'protein_mutation_score': data['protein_mutation_score']
    })

df_history = pd.DataFrame(plot_data)

# Define the interactive plotting function that takes rmsd_threshold as input
def update_plots(rmsd_threshold):
    if df_history.empty:
        print("No data in history to plot.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Protein Properties Analysis for Unconstrained Sequences', fontsize=16)

    # Subplot 1: Sequence Identity vs RMSD (static, but within the update function to redraw with the interactive plot)
    sns.scatterplot(data=df_history, x='sequence_id', y='rmsd', ax=axes[0])
    axes[0].set_title('Sequence Identity vs. RMSD')
    axes[0].set_xlabel('Sequence Identity (%)')
    axes[0].set_ylabel('RMSD (Å)')
    axes[0].set_xlim(0, 100)
    axes[0].set_ylim(0, 15)
    axes[0].grid(True, linestyle='--', alpha=0.7)

    # Subplot 2: Fraction of Sequences with RMSD < Threshold (interactive)
    # Bin sequence identity into 5% wide bins
    bins = np.arange(0, 101, 5)
    # Ensure 'sequence_id_bin' is created within the function if it's not already existing for interactive updates
    # Make a copy to avoid SettingWithCopyWarning if df_history is modified outside this function scope later
    df_temp = df_history.copy()
    df_temp['sequence_id_bin'] = pd.cut(df_temp['sequence_id'], bins=bins, right=False)

    # Calculate fraction below threshold for each sequence_id_bin
    grouped_data = df_temp.groupby('sequence_id_bin', observed=False).apply(lambda x:
        pd.Series({
            'num_total': len(x),
            'num_below_threshold': (x['rmsd'] < rmsd_threshold).sum() # Use the dynamic threshold here
        }), include_groups=False
    ).reset_index()

    grouped_data['fraction_below_threshold'] = grouped_data['num_below_threshold'] / grouped_data['num_total']

    # Calculate standard error
    grouped_data['std_err'] = np.sqrt(
        grouped_data['fraction_below_threshold'] * (1 - grouped_data['fraction_below_threshold']) /
        grouped_data['num_total']
    )

    # Fill NaN values specifically in the numerical columns
    grouped_data['fraction_below_threshold'] = grouped_data['fraction_below_threshold'].fillna(0)
    grouped_data['std_err'] = grouped_data['std_err'].fillna(0)

    # Convert bin labels to midpoints for plotting
    grouped_data['sequence_id_midpoint'] = grouped_data['sequence_id_bin'].apply(lambda x: x.mid)

    selected_grouped_data = grouped_data[grouped_data['num_total'] > 0]
    axes[1].errorbar(
        x=selected_grouped_data['sequence_id_midpoint'],
        y=selected_grouped_data['fraction_below_threshold'],
        yerr=selected_grouped_data['std_err'],
        fmt='o', # Marker style
        capsize=5, # Cap size for error bars
        label=f'Fraction of sequences with RMSD < {rmsd_threshold:.1f}Å' # Update label
    )

    axes[1].set_title(f'Fraction of Sequences with RMSD < {rmsd_threshold:.1f}Å vs. Sequence Identity') # Update title
    axes[1].set_xlabel('Sequence Identity (%)')
    axes[1].set_ylabel('Fraction of Sequences')
    axes[1].set_xlim(0, 100)
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(True, linestyle='--', alpha=0.7)
    axes[1].legend()

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Create the slider widget for RMSD threshold
rmsd_threshold_slider = widgets.FloatSlider(
    value=2.0,  # Default value, can be adjusted
    min=0.0,
    max=10.0,
    step=0.1,
    description='RMSD Threshold (Å):',
    continuous_update=True,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

# Use interact to connect the slider to the plotting function
# This will display the slider and the plots below it
widgets.interact(update_plots, rmsd_threshold=rmsd_threshold_slider);


## Report Questions

A reasonable number of sequences to run at a given sequence identity is about 20 or more, to get sufficient statistics.

### Q1. At what sequence identity do you find that less than half of the generated sequences have the same fold as the starting sequence, judged by the RMSD? Attach your plots (screenshot is fine).  

### Q2. It's a general rule that "Any 2 protein sequences you compare have the same fold if they have more than 30% sequence identity." This is clearly wrong, given what you observe here. What is the correct statement?

# Generating constrained mutations

No lets generate **constrained** mutations by iteratively mutating and folding using ESM-fold. This will emulate the force of evolutionary pressure, by trying to add mutations on top of each other, only accepting those with low RMSD. If the RMSD is too high compared to the increase in mutation, this mutation is rejected and another tried instead. This essentially rewards low Sequence ID and high similarity, effectively trying to explore as far away as possible in this region of fold-space.

> **METAPHOR**: In our desert of sequences, all grains of sand that are red enough (similar to starting), form a kind of sub-space. The constrained mutations are in essence searching around their current position of red grains, trying to move away from the starting position. This sub-space of red grains of sand is fold-space.



The unconstrained mutations showed us how far away we could get before the probability gets very low for finding the same fold.

##Ready to evolve!

So the procedure **that the code will do** (you just specify how many times to do it) is to

1. Suggest a random mutation to a random position in the sequence
2. Use a protein language model to predict the structure
3. Compute a score based on sequence similarity and structure similarity (RMSD)
4. Accept if the score is better, otherwise reject

This goes on for as long as you specify. For a trial run, we suggest maybe 25 *iterations*.  

### Settings
- You definitely want to **save history**
- You can run more than one mutation path, by changing the **history filename**. To continue a path, make sure you have the right filename and tick **load history**
 - If you tick **use conservative mutations**, only "reasonable" amino acid mutations will be attmepted.



In [ ]:
import numpy as np
import numpy.random as random
import os, re, requests # Import necessary modules for OmegaFold and file handling
from string import ascii_uppercase, ascii_lowercase # Import for chain naming
import hashlib # Import for hashing sequence
import pickle # Import for saving/loading
import gc

num_iterations = 200 # @param {type:"integer"}
mutations_per_iteration = 1 # @param {type:"integer"} # New parameter for protein mutations
save_history = True # @param {type:"boolean"}
history_filename = "constrained.pkl" # @param {type:"string"}
load_history = False # @param {type:"boolean"}
use_precomputed = False
conservative_mutations = False # @param {type:"boolean"}
use_omega = False  # @param {type:"boolean"}

conservative_substitutions = {
    'A': ['G', 'S', 'T'], # Alanine (small, nonpolar)
    'R': ['K', 'H'],     # Arginine (basic, positive)
    'N': ['Q', 'S', 'T'], # Asparagine (polar, uncharged)
    'D': ['E'],          # Aspartic Acid (acidic, negative)
    'C': ['S'],          # Cysteine (sulfur-containing)
    'Q': ['N', 'H', 'K', 'R', 'E', 'D'], # Glutamine (polar, uncharged) - larger group due to flexibility
    'E': ['D', 'Q', 'N', 'K', 'R', 'H'], # Glutamic Acid (acidic, negative) - larger group due to flexibility
    'G': ['A', 'S'],     # Glycine (small, nonpolar)
    'H': ['R', 'K', 'N', 'Q'], # Histidine (basic, positive, aromatic)
    'I': ['L', 'V', 'M'], # Isoleucine (hydrophobic)
    'L': ['I', 'V', 'M'], # Leucine (hydrophobic)
    'K': ['R', 'H', 'Q', 'N'], # Lysine (basic, positive)
    'M': ['L', 'I', 'V'], # Methionine (hydrophobic, sulfur-containing)
    'F': ['Y', 'W'],     # Phenylalanine (aromatic, hydrophobic)
    'P': [],             # Proline (cyclic, often disrupts structure - few conservative substitutions)
    'S': ['T', 'N', 'Q', 'A', 'G', 'C'], # Serine (polar, uncharged, hydroxyl)
    'T': ['S', 'N', 'Q', 'A', 'G'], # Threonine (polar, uncharged, hydroxyl)
    'W': ['Y', 'W'],     # Tryptophan (aromatic, hydrophobic, large)
    'Y': ['F', 'W'],     # Tyrosine (aromatic, hydrophobic, hydroxyl)
    'V': ['I', 'L', 'M']  # Valine (hydrophobic)
}

constrained_history_file = "constrained_protein_history.pkl"

all_amino_acids = list("ACDEFGHIKLMNPQRSTVWY") # Global constant now

def mutate_protein_sequence(protein_seq: str, num_mutations: int, conservative_only: bool) -> str:
    """
    Generates a new protein sequence with a specified number of mutations.

    Args:
        protein_seq: The original protein sequence string.
        num_mutations: The number of random mutations to introduce.
        conservative_only: If True, only generates conservative substitutions.

    Returns:
        A new protein sequence string with the specified number of mutations.
    """
    mutated_protein_list = list(protein_seq)
    seq_len = len(mutated_protein_list)

    if num_mutations <= 0 or seq_len == 0: # Handle edge cases
        return protein_seq

    # Select unique positions to mutate
    positions_to_mutate = random.choice(np.arange(seq_len), size=min(num_mutations, seq_len), replace=False)

    for pos in positions_to_mutate:
        current_aa = mutated_protein_list[pos]
        possible_new_aas = []

        if conservative_only:
            if current_aa in conservative_substitutions:
                possible_new_aas = [aa for aa in conservative_substitutions[current_aa] if aa != current_aa]
        else:
            possible_new_aas = [aa for aa in all_amino_acids if aa != current_aa]

        if possible_new_aas:
            mutated_protein_list[pos] = random.choice(possible_new_aas)

    return "".join(mutated_protein_list)

# Function to get hash for jobname
def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()
alphabet_list = list(ascii_uppercase+ascii_lowercase)

if use_precomputed:
  history_filename="mutation_history_znf.pkl"
  load_history = True
  #num_iterations=np.max([num_iterations,200])
if not os.path.isfile(history_filename):
  !wget -q -O {history_filename} https://raw.githubusercontent.com/bforsbe/SK2534/main/mutation_history_znf.pkl

# Get the original protein sequence from the widget in cell 7c66c75a
# It's assumed this cell (7c66c75a) has been run and its widgets are accessible
try:
    # Accessing widget value directly if original_protein_sequence_text is in global scope
    original_protein_sequence = original_protein_sequence_text.value.upper().strip()
except NameError:
    print("Error: 'original_protein_sequence_text' widget not found. Please ensure cell 7c66c75a is run.")
    original_protein_sequence = ""

if not original_protein_sequence:
    print("No original protein sequence found. Please define it in cell 7c66c75a and run it.")
    # Fallback to a default if absolutely necessary, but user should define
    original_protein_sequence = "YKCGLCERSFVEKSALSRHQRVHKN" # Using the default from 7c66c75a


# --- MODIFIED: Load history if exists, otherwise initialize new history ---
if load_history and os.path.exists(history_filename):
    try:
        with open(history_filename, 'rb') as f:
            history = pickle.load(f)
        print(f"Loaded existing history from {history_filename}")

        # Ensure original_protein_sequence in loaded history matches current one
        if history.get('original_protein_sequence') != original_protein_sequence:
            print("Warning: Original protein sequence in loaded history does not match current input. Using loaded history's original sequence.")
            original_protein_sequence = history['original_protein_sequence'] # Use the original from the loaded history

    except Exception as e:
        print(f"Error loading existing history from {history_filename}: {e}. Starting new history.")
        # Fallback to new history if loading fails
        history = {
            'best_dna_sequence': "", # Placeholder, as we no longer directly mutate DNA
            'best_protein_sequence': original_protein_sequence,
            'best_total_score': -1,
            'accepted_mutation_scores': [],
            'accepted_sim_scores': [],
            'accepted_total_scores': [],
            'accepted_iterations': [],
            'all_mutation_scores': [],
            'all_sim_scores': [],
            'all_total_scores': [],
            'all_iterations': [],
            'all_tried_protein_sequences': [original_protein_sequence],
            'completed_iters': 0,
            'original_protein_sequence': original_protein_sequence # Store original protein for context
        }
else: # Either load_history is False, or the file doesn't exist
    print(f"Starting new history (load_history={load_history}, file_exists={os.path.exists(history_filename)}).")
    history = {
        'best_dna_sequence': "", # Placeholder, as we no longer directly mutate DNA
        'best_protein_sequence': original_protein_sequence,
        'best_total_score': -1,
        'best_rmsd': 999,
        'best_sequence_identity': 100,
        'accepted_mutation_scores': [],
        'accepted_sim_scores': [],
        'accepted_total_scores': [],
        'accepted_iterations': [],
        'all_mutation_scores': [],
        'all_sim_scores': [],
        'all_total_scores': [],
        'all_iterations': [],
        'all_tried_protein_sequences': [original_protein_sequence],
        'completed_iters': 0,
        'original_protein_sequence': original_protein_sequence # Store original protein for context
    }

full_history = {}
if os.path.exists(constrained_history_file):
    with open(constrained_history_file, 'rb') as f:
        full_history = pickle.load(f)
    print(f"Loaded existing history from {constrained_history_file} with {len(full_history)} entries.")
else:
    print(f"No existing history found. Creating new history file: {constrained_history_file}")

# Extract current state from history for the loop
best_protein_sequence = history['best_protein_sequence']
best_total_score = history['best_total_score']
best_rmsd = history['best_rmsd']
best_sequence_identity = history['best_sequence_identity']
accepted_mutation_scores = history['accepted_mutation_scores']
accepted_sim_scores = history['accepted_sim_scores']
accepted_total_scores = history['accepted_total_scores']
accepted_iterations = history['accepted_iterations']
all_mutation_scores = history['all_mutation_scores']
all_sim_scores = history['all_sim_scores']
all_total_scores = history['all_total_scores']
all_iterations = history['all_iterations']
all_tried_protein_sequences = history['all_tried_protein_sequences']
completed_iters = history['completed_iters']
# END MODIFIED SECTION


while completed_iters < num_iterations:
    # Generate a mutated protein sequence directly
    single_mutated_protein_seq = mutate_protein_sequence(best_protein_sequence, mutations_per_iteration, conservative_only=conservative_mutations)

    # Check if protein sequence has been tried before
    if single_mutated_protein_seq in all_tried_protein_sequences:
        completed_iters += 1 # Count as an iteration even if skipped to avoid infinite loops
        continue

    all_tried_protein_sequences.append(single_mutated_protein_seq)

    # Predict structure using OmegaFold
    # Adapt the OmegaFold execution code from cell CFCwEAa2oZEN
    #try:
    # Generate a unique jobname based on iteration and a hash of the sequence
    jobname = f"iter_{completed_iters}_{get_hash(single_mutated_protein_seq)[:5]}"
    ID = jobname
    seqs = single_mutated_protein_seq.split(":")
    lengths = [len(s) for s in seqs]

    # Write fasta file
    with open(f"{ID}.fasta","w") as out:
        out.write(f">{ID}\n{single_mutated_protein_seq}\n")

    # Run OmegaFold command
    # Use appropriate parameters, likely inherited from the successful OmegaFold cell
    subbatch_size = 1000 # Assuming single chain, length < 500 from previous cell
    num_cycle = 2 # Assuming from previous cell
    offset_rope = False # Assuming from previous cell
    device = "cuda" if torch.cuda.is_available() else "cpu" # Get device from setup cell

    if use_omega:
        # Execute OmegaFold prediction
        # Capture output to keep the notebook clean
        with io.capture_output() as captured:
            %shell python OmegaFold/main.py --offset_rope={offset_rope} --device={device} --subbatch_size={subbatch_size} --num_cycle={num_cycle} {ID}.fasta .

        # Define the predicted PDB file name
        single_mutated_pdb_file = f"{ID}.pdb"

        # Read the predicted PDB file
    else:
      # Execute ESMFold prediction
      if "model" not in dir() or model_name != model_name_:
        if "model" in dir():
          # delete old model from memory
          del model
          gc.collect()
          if torch.cuda.is_available():
            torch.cuda.empty_cache()

        model = torch.load(model_name, weights_only=False)
        model.eval().cuda().requires_grad_(False)
        model_name_ = model_name

      # optimized for Tesla T4
      model.set_chunk_size(128)

      torch.cuda.empty_cache()
      with io.capture_output() as captured:
          sequence = "".join(open(f"{ID}.fasta").read().splitlines()[1:])

          output = model.infer(sequence, num_recycles=3)

          pdb_str = model.output_to_pdb(output)[0]

          single_mutated_pdb_file = f"{ID}.pdb"
          with open(single_mutated_pdb_file, "w") as f:
              f.write(pdb_str)

    with open(single_mutated_pdb_file, 'r') as f:
        single_mutated_pdb_str = f.read()

    # Renumber PDB if necessary (using the function from CFCwEAa2oZEN)
    single_mutated_pdb_str = renum_pdb_str(single_mutated_pdb_str, Ls=lengths)
    # Save the renumbered PDB string to a file for alignment
    with open(single_mutated_pdb_file, "w") as out:
          out.write(single_mutated_pdb_str)


    # Align to reference and calculate RMSD (using the function from c51BDdsXe68l)
    aligned_file_single, single_mutated_rmsd_val = align_pred_to_ref(single_mutated_pdb_str, "reference.pdb", out_file=f"{ID}_aligned.pdb")


    # Convert RMSD to similarity score (using the function from c51BDdsXe68l)
    single_mutated_pred_sim_score = rmsd_to_similarity(single_mutated_rmsd_val)

    # Calculate mutation score using protein_mutation_score_direct
    single_current_mutation_score = protein_mutation_score_direct(original_protein_sequence, single_mutated_protein_seq)

    # Calculate total score
    single_current_total_score = single_current_mutation_score * single_mutated_pred_sim_score

    # Store scores for ALL tried sequences
    all_iterations.append(completed_iters + 1)
    all_mutation_scores.append(single_current_mutation_score)
    all_sim_scores.append(single_mutated_pred_sim_score)
    all_total_scores.append(single_current_total_score)

    # Print trial sequence and total score
    print(f"Iteration {completed_iters + 1}: Trial Sequence: {single_mutated_protein_seq}, Total Score: {single_current_total_score:.3f}")

    # Acceptance criteria: accept if total score improves or prediction is very similar
    if single_current_total_score > best_total_score or single_mutated_pred_sim_score > 0.9:
        best_total_score = single_current_total_score
        #best_dna_sequence = single_mutated_dna_seq # No longer directly tracking DNA
        best_protein_sequence = single_mutated_protein_seq
        best_rmsd = single_mutated_rmsd_val
        best_sequence_identity = single_mutated_pred_sim_score

        # Store scores for ACCEPTED sequences
        accepted_iterations.append(completed_iters + 1)
        accepted_mutation_scores.append(single_current_mutation_score)
        accepted_sim_scores.append(single_mutated_pred_sim_score)
        accepted_total_scores.append(single_current_total_score)

        #print(f"  Accepted mutation. New best total score: {best_total_score:.3f}")

    completed_iters += 1

    # Save history
    if save_history:
        history = {
            'best_dna_sequence': "", # Placeholder, as we no longer directly mutate DNA
            'best_protein_sequence': best_protein_sequence,
            'best_total_score': best_total_score,
            'best_rmsd': best_rmsd,
            'best_sequence_identity': best_sequence_identity,
            'accepted_mutation_scores': accepted_mutation_scores,
            'accepted_sim_scores': accepted_sim_scores,
            'accepted_total_scores': accepted_total_scores,
            'accepted_iterations': accepted_iterations,
            'all_mutation_scores': all_mutation_scores,
            'all_sim_scores': all_sim_scores,
            'all_total_scores': all_total_scores,
            'all_iterations': all_iterations, # This line was previously commented out and is now uncommented.
            'all_tried_protein_sequences': all_tried_protein_sequences,
            'completed_iters': completed_iters,
            'original_protein_sequence': original_protein_sequence # Store original protein for context
        }
        with open(history_filename, 'wb') as f:
            pickle.dump(history, f)
        # Also save as .npy for easier access (optional)
        np.save(history_filename.replace('.pkl', '.npy'), history)


#except Exception as e:
#    print(f"Iteration {completed_iters + 1}: Error processing mutation: {e}. Skipping.")
#    completed_iters += 1 # Increment even if error occurs to avoid infinite loops if errors persist
#    continue # Continue to the next iteration if an error occurs

print("\nIterative mutation process complete.")
# Removed printing best_dna_sequence as it's no longer directly tracked
print(f"Best Mutated Protein Sequence: {best_protein_sequence}")
print(f"RMSD of best mutated to starting: {best_rmsd:.3f}")
print(f"Sequence Identity of best mutated to starting:: {100*best_sequence_identity:.2f}%")

# Recalculate final scores for the best sequence
# Ensure both are standard Python strings and stripped
final_original_protein_seq_for_score = original_protein_sequence.strip()
final_best_protein_sequence_for_score = str(best_protein_sequence).strip()

# Add a sanity check print to help debug if the issue persists
#print(f"DEBUG: Final original protein seq length: {len(final_original_protein_seq_for_score)}, Type: {type(final_original_protein_seq_for_score)}")
#print(f"DEBUG: Final best protein seq length: {len(final_best_protein_sequence_for_score)}, Type: {type(final_best_protein_sequence_for_score)}")
#if len(final_original_protein_seq_for_score) != len(final_best_protein_sequence_for_score):
#    #print("DEBUG: Length mismatch detected before calling protein_mutation_score_direct!")
#else:
#    #print("DEBUG: Lengths are equal, proceeding to protein_mutation_score_direct.")

best_mutation_score = protein_mutation_score_direct(final_original_protein_seq_for_score, final_best_protein_sequence_for_score)
# To get the final RMSD and prediction similarity score for the best sequence,
# you would need to re-run the prediction and alignment for best_protein_sequence
# or store these values when the best sequence is updated.
# For simplicity, we'll just report the best total score found.
print(f"Best Total Score Found: {best_total_score:.3f}")
print(f"Total number of tried protein sequences: {len(all_tried_protein_sequences)}")
print(f"Total number of accepted mutations: {len(accepted_total_scores)}")

### Comparison between Unconstrained and Constrained Mutations

Lets now visualize the unconstrained mutations (where mutations were generated randomly) and the constrained mutations (where evolutionary pressure was applied). The plot below shows the relationship between sequence identity and RMSD (Root Mean Square Deviation), as well as the fraction of sequences that fold correctly (based on an RMSD threshold) across different ranges of sequence identity.

In [ ]:
# @title Plot constrained and unconstrained samples
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import pickle

# Assuming df_history is already loaded from cell 007a0fad
# If df_history is not available, it can be reloaded here:
# unconstrained_history_file = "unconstrained_protein_history.pkl"
# history_unconstrained = {}
# if os.path.exists(unconstrained_history_file):
#     with open(unconstrained_history_file, 'rb') as f:
#         history_unconstrained = pickle.load(f)
# plot_data_unconstrained = []
# for seq, data in history_unconstrained.items():
#     plot_data_unconstrained.append({
#         'sequence_id': data['seq_id'],
#         'rmsd': data['rmsd'],
#         'total_score': data['total_score'],
#         'protein_mutation_score': data['protein_mutation_score']
#     })
# df_history = pd.DataFrame(plot_data_unconstrained)

# Load constrained history
history_file_constrained = "constrained.pkl" #@param
history_constrained = {}
if os.path.exists(history_file_constrained):
    with open(history_file_constrained, 'rb') as f:
        history_constrained = pickle.load(f)
    print(f"Loaded constrained history from {history_file_constrained}")
else:
    print(f"No constrained history file found at {history_file_constrained}. Ensure the evolution cell (kHO4EmWdi9CC) is run first.")

constrained_plot_data = []
if history_constrained and 'all_mutation_scores' in history_constrained and 'all_sim_scores' in history_constrained:
    # Convert sim_scores to rmsd_val. Assuming sigma=2.0 from rmsd_to_similarity function
    # The inverse of rmsd_to_similarity(rmsd, sigma) is: rmsd = -sigma * log(similarity)
    # If similarity is a percentage, it should be sim_score / 100
    # The function rmsd_to_similarity used in 2ecc1fbf is exp(-rmsd / sigma)
    # So, rmsd = -sigma * log(similarity)
    # Given pred_sim_score = rmsd_to_similarity(rmsd_val) and sigma = 2.0 (from cell 2ecc1fbf),
    # then rmsd_val = -2.0 * np.log(pred_sim_score)
    # if sim_score is percentage like in the context, then sim_score/100.
    # Therefore, rmsd_val = -2.0 * np.log(sim_score / 100) if sim_score > 0 else np.nan
    # This is slightly different from the previous code, which was '-2.0 * np.log(sim_score)'.
    # The previous code seemed to assume sim_score was already scaled (0-1). Let's assume sim_score is 0-100 percentage based on common usage.

    for i in range(len(history_constrained['all_mutation_scores'])):
        seq_id = 100.0 - history_constrained['all_mutation_scores'][i] # Mutation score is 100 - identity
        sim_score = history_constrained['all_sim_scores'][i] * 100.0 # Scale sim_score to percentage 0-100

        # Correct RMSD calculation using the inverse of rmsd_to_similarity (sigma=2.0, sim_score in 0-100 range)
        rmsd_val = -2.0 * np.log(sim_score / 100.0) if sim_score > 0 else np.nan

        constrained_plot_data.append({
            'sequence_id': seq_id,
            'rmsd': rmsd_val,
            'total_score': history_constrained['all_total_scores'][i],
            'protein_mutation_score': history_constrained['all_mutation_scores'][i],
            'type': 'Constrained'
        })
df_constrained = pd.DataFrame(constrained_plot_data).dropna(subset=['rmsd'])

# Prepare unconstrained data (from df_history)
df_unconstrained = df_history.copy()
df_unconstrained['type'] = 'Unconstrained'

# Combine dataframes for unified plotting
df_combined = pd.concat([df_unconstrained, df_constrained])


def plot_comparison_with_slider(rmsd_threshold):
    if df_combined.empty:
        print("No data to plot.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Analysis of Sequence Properties (Unconstrained vs. Constrained)', fontsize=16)

    # Subplot 1: Sequence Identity vs RMSD Scatter Plot
    sns.scatterplot(data=df_combined, x='sequence_id', y='rmsd', hue='type', ax=axes[0], alpha=0.6, s=50)
    axes[0].set_title('Sequence Identity vs. RMSD')
    axes[0].set_xlabel('Sequence Identity (%)')
    axes[0].set_ylabel('RMSD (Å)')
    axes[0].set_xlim(0, 100)
    # Set y-limit to 0 to avoid negative RMSD values, but let it float upwards
    axes[0].set_ylim(bottom=0)
    axes[0].grid(True, linestyle='--', alpha=0.7)
    axes[0].legend(title='Mutation Type')

    # Subplot 2: Fraction of Sequences with RMSD < Threshold (Error Bar Plot)
    bins = np.arange(0, 101, 5)  # 5% wide bins for sequence identity

    grouped_data_list = []
    for mutation_type in df_combined['type'].unique():
        df_temp = df_combined[df_combined['type'] == mutation_type].copy()
        df_temp['sequence_id_bin'] = pd.cut(df_temp['sequence_id'], bins=bins, right=False)

        grouped_data = df_temp.groupby('sequence_id_bin', observed=False).apply(lambda x:
            pd.Series({
                'num_total': len(x),
                'num_below_threshold': (x['rmsd'] < rmsd_threshold).sum()
            }), include_groups=False
        ).reset_index()

        grouped_data['fraction_below_threshold'] = grouped_data['num_below_threshold'] / grouped_data['num_total']
        # Calculate standard error
        grouped_data['std_err'] = np.sqrt(
            grouped_data['fraction_below_threshold'] * (1 - grouped_data['fraction_below_threshold']) /
            grouped_data['num_total']
        )
        grouped_data['fraction_below_threshold'] = grouped_data['fraction_below_threshold'].fillna(0)
        grouped_data['std_err'] = grouped_data['std_err'].fillna(0)
        grouped_data['sequence_id_midpoint'] = grouped_data['sequence_id_bin'].apply(lambda x: x.mid)
        grouped_data['type'] = mutation_type  # Add type column to grouped data

        grouped_data_list.append(grouped_data)

    df_grouped_combined = pd.concat(grouped_data_list)

    # Use lineplot to draw the lines and then draw error bars manually
    sns.lineplot(data=df_grouped_combined[df_grouped_combined['num_total'] > 0],
                 x='sequence_id_midpoint', y='fraction_below_threshold',
                 hue='type', marker='o', ax=axes[1], errorbar=None)  # errorbar=None to draw error bars manually

    # Manual error bars to allow different colors per line
    palette = sns.color_palette()
    for i, mutation_type in enumerate(df_grouped_combined['type'].unique()):
        data_subset = df_grouped_combined[(df_grouped_combined['type'] == mutation_type) & (df_grouped_combined['num_total'] > 0)]
        color = palette[i]
        axes[1].errorbar(x=data_subset['sequence_id_midpoint'],
                         y=data_subset['fraction_below_threshold'],
                         yerr=data_subset['std_err'],
                         fmt='none', capsize=3, color=color, alpha=0.7)


    axes[1].set_title(f'Fraction of Sequences with RMSD < {rmsd_threshold:.1f}Å vs. Sequence Identity')
    axes[1].set_xlabel('Sequence Identity (%)')
    axes[1].set_ylabel('Fraction of Sequences')
    axes[1].set_xlim(0, 100)
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(True, linestyle='--', alpha=0.7)
    axes[1].legend(title='Mutation Type')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Create the slider widget for RMSD threshold
rmsd_threshold_slider_comparison = widgets.FloatSlider(
    value=2.0,
    min=0.0,
    max=10.0,
    step=0.1,
    description='RMSD Threshold (Å):',
    continuous_update=True,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

print("Configure RMSD threshold for the comparison plot:")
widgets.interact(plot_comparison_with_slider, rmsd_threshold=rmsd_threshold_slider_comparison);

## Report Questions  

### Q3. What is the lowest sequence identity you can reach that has the same fold as the original sequence? Provide
- sequence
- sequence identity to original
- RMSD

### Q4. Compare the constrained mutations to the unconstrained. What differneces do you observe in the plots, and why?

### Q5. Explain in your own words why constrained mutations can get lower sequence identity than random mutations seem to be able to within a reasonable timespan.

# Sequence similarity analysis
The constrained mutation (evolution) resulted in a sequence that may actually exist in nature - who knows, right? **Well let's check.**

The following compiles an MSA of the mutated sequence you got along some 3000 protein sequences from nature that have the same fold. They are all quite different, so maybe our mutated sequence or something close to it is in there?

These 3000 sequences were found using a tool called foldseek. It's a pretty cool tool, but we won't go into details on it here, but talk about it late rin the course). Also, instead of using our code to make the MSA, we will use Clustal.

**NOTE** You should not be surprised if the 3k MSA takes a bit of time - you have seen how computationally demanding this gets.

**NOTE** You have to put in the mutated sequence manually!

In [ ]:
#@title Install Clustal

%pip install biopython
!apt-get update -y
!apt-get install -y clustalw

In [ ]:
#@title Make the MSA

import requests
import io
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import Bio.Align
import subprocess # Added import for subprocess

# Define the original and query sequence strings
original_sequence = "YKCGLCERSFVEKSALSRHQRVHKN" #@param {type:"string"}
mutated_sequence = "FECHQCQYFSYKRVLMRVHQVMHIR" #@param {type:"string"}

# Define the URL of the FASTA file and the local filename
url = "https://raw.githubusercontent.com/bforsbe/SK2534/main/aligned_output.fasta"
local_fasta_file = "template_aligned.fasta"

# Download the content of the file and save it locally if it doesn't exist
try:
    with open(local_fasta_file, "r") as f:
        print(f"Using existing template file: {local_fasta_file}")
except FileNotFoundError:
    print(f"Downloading template file from: {url}")
    response = requests.get(url)
    response.raise_for_status() # Raise an exception for bad status codes
    with open(local_fasta_file, "w") as f:
        f.write(response.text)
    print(f"Template file downloaded and saved as: {local_fasta_file}")

# Read the sequences from the local FASTA file using SeqIO.parse
sequences = list(SeqIO.parse(local_fasta_file, "fasta"))

# Print the number of sequences loaded to verify
print(f"Number of sequences loaded from template: {len(sequences)}")

original_sequence_string = original_sequence
query_sequence_string = mutated_sequence
# Create SeqRecord objects for the original and query sequences with fixed IDs
original_sequence = SeqRecord(Seq(original_sequence_string), id="Original_Sequence", description="")
query_sequence = SeqRecord(Seq(query_sequence_string), id="Query_Sequence", description="")

# Create a new list of sequences including the original sequences from the template, the original, and the query
sequences_with_original_and_query = sequences + [original_sequence, query_sequence]

# Print the updated number of sequences to verify
print(f"Updated number of sequences (including original and mutated): {len(sequences_with_original_and_query)}")

# Modify sequence IDs to be unique and clear descriptions for the list including the query and original
for i, seq in enumerate(sequences_with_original_and_query):
    # Only modify IDs if they are not the original or query sequences
    if seq.id != "Original_Sequence" and seq.id != "Query_Sequence":
        seq.id = f"{seq.id}_{i}"
    seq.description = "" # Clear description to avoid issues with some tools

# Define a filename for the temporary FASTA file
temp_in_file = "temp_sequences_for_alignment.fasta"

# Write the modified sequences to the temporary FASTA file
SeqIO.write(sequences_with_original_and_query, temp_in_file, "fasta")

# Print confirmation message
print(f"Prepared sequences (including original and query) and saved to {temp_in_file}")

from Bio import AlignIO
# The import from Bio.Align.Applications is removed as it caused ModuleNotFoundError
# from Bio.Align.Applications import ClustalwCommandline

# Define the name for the output alignment file
out_file_aln = "aligned_with_query.aln"
out_file_fasta = "aligned_with_query.fasta"

# Create a ClustalwCommandline object (replaced with direct subprocess call)
# Specify the path to the clustalw executable if it's not in your PATH
print("Running Clustal...")

# clustalw_cline = ClustalwCommandline("clustalw", infile=temp_in_file, outfile=out_file_aln)
# stdout, stderr = clustalw_cline()

# Using subprocess to call clustalw directly
try:
    result = subprocess.run(
        ["clustalw", "-infile=" + temp_in_file, "-outfile=" + out_file_aln],
        capture_output=True, text=True, check=True
    )
    stdout = result.stdout
    stderr = result.stderr
    #print(stdout)
    if stderr: # Print stderr only if there's content
        print(f"ClustalW encountered warnings or non-critical errors:\n{stderr}")
except subprocess.CalledProcessError as e:
    print(f"Error running ClustalW: {e}")
    print(f"Stdout: {e.stdout}")
    print(f"Stderr: {e.stderr}")
    raise # Re-raise the exception to stop execution if clustalw failed critically

# Print confirmation message
print(f"Multiple sequence alignment performed and saved to {out_file_aln}")

# Convert aln -> fasta using AlignIO (still needed for writing)
alignment_aln = AlignIO.read(out_file_aln, "clustal")
AlignIO.write(alignment_aln, out_file_fasta, "fasta")

Now we look through these 3000 sequences and find the best match to our maximum mutated sequence.


In [ ]:
#@title Find and show closest match
import Bio
from Bio.Align import MultipleSeqAlignment
from Bio import AlignIO

# Define the name of the alignment file output by ClustalW
alignment_file = out_file_aln

# Read the alignment file using AlignIO.read with 'clustal' format
try:
    alignment = AlignIO.read(alignment_file, "clustal")

    # Print the alignment object to verify
    print("Alignment object:")
    print(alignment)

except FileNotFoundError:
    print(f"Error: The file {alignment_file} was not found.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

# Find the query and original sequences in the alignment using their fixed IDs
query_sequence_id = "Query_Sequence"
original_sequence_id = "Original_Sequence"
query_seq = None
original_seq = None

for record in alignment:
    if record.id == query_sequence_id:
        query_seq = record
    elif record.id == original_sequence_id:
        original_seq = record

if query_seq is None:
    print(f"Error: Query sequence with ID '{query_sequence_id}' not found in the alignment.")
else:
    max_identity = 0.0
    best_match_id = None
    best_match_seq = None

    # Iterate through each sequence in the alignment
    for record in alignment:
        # Skip the query and original sequences themselves when finding best match
        if record.id == query_sequence_id or record.id == original_sequence_id:
            continue

        # Calculate sequence identity
        identical_residues = 0
        aligned_length = 0
        for i in range(alignment.get_alignment_length()):
            query_residue = query_seq[i]
            current_residue = record[i]

            # Only consider positions where neither sequence has a gap
            if query_residue != '-' and current_residue != '-':
                aligned_length += 1
                if query_residue == current_residue:
                    identical_residues += 1

        # Avoid division by zero if there are no aligned positions
        if aligned_length > 0:
            identity = (identical_residues / aligned_length) * 100
            if identity > max_identity:
                max_identity = identity
                best_match_id = record.id
                best_match_seq = record
        else:
            identity = 0.0 # Or handle as appropriate if no comparable positions

    print(f"Maximum sequence identity of the query sequence against others (excluding original): {max_identity:.2f}%")
    if best_match_id:
        print(f"Sequence with the highest identity: {best_match_id}")

        # Display the alignment of the query, original, and the best match
        print("\nAlignment of Mutated Sequence, Original Sequence, and Best Match:")
        print("-" * 60)

        # Filter out all-dash columns for the three sequences being printed
        original_seq_filtered_str = ""
        query_seq_filtered_str = ""
        best_match_seq_filtered_str = ""

        # Determine the maximum length of the sequence IDs for alignment
        max_id_length = max(len(query_seq.id), len(original_seq.id) if original_seq else 0, len(best_match_id))

        for i in range(alignment.get_alignment_length()):
            current_original_residue = original_seq[i] if original_seq else '-'
            current_query_residue = query_seq[i]
            current_best_match_residue = best_match_seq[i]

            # Check if all three residues at this position are dashes
            if not (current_original_residue == '-' and current_query_residue == '-' and current_best_match_residue == '-') :
                original_seq_filtered_str += current_original_residue
                query_seq_filtered_str += current_query_residue
                best_match_seq_filtered_str += current_best_match_residue

        if original_seq:
             print(f"{original_seq_filtered_str} {original_seq.id.ljust(max_id_length)}")
        print(f"{query_seq_filtered_str} {"Mutated sequence".ljust(max_id_length)}")
        print(f"{best_match_seq_filtered_str} {best_match_id.ljust(max_id_length)}")

## Report Questions

### Q7. The original sequence was a human zinc-binding domain, what was your closest match to the mutated sequence, and what was the sequence identity to it?

### Q8. Compare the sequence identity (distance) between your original, mutated, and closest match. Comment on whether you find this to be expected, unexpected, unintuitive, or any other reflection.

Finally, lets look at the sequewnce logo for the MSA of all 3000 natural sequences.

In [ ]:
!pip install logomaker

In [ ]:
#@title Create sequence logo
from Bio import AlignIO
from collections import Counter
import pandas as pd
import logomaker
import matplotlib.pyplot as plt

# Read the alignment (FASTA or ALN)
alignment = AlignIO.read("aligned_with_query.aln", "clustal")

num_seqs = len(alignment)
alignment_length = alignment.get_alignment_length()
print("Number of sequences:", num_seqs)
print("Alignment length:", alignment_length)

# Build consensus sequence (preserving all columns)
consensus = ""
degenerate_consensus = ""
counts_matrix = []  # for logo

for col_idx in range(alignment_length):
    column = alignment[:, col_idx]             # get column as string
    counts = Counter(res for res in column if res != "-")  # ignore gaps
    counts_matrix.append({aa: counts.get(aa, 0) for aa in "ACDEFGHIKLMNPQRSTVW"})

    if counts:
        # Majority vote for consensus
        consensus += counts.most_common(1)[0][0]
        # Degenerate: include all residues that appear in column
        residues = sorted(counts.keys())
        degenerate_consensus += "[" + "".join(residues) + "]" if len(residues) > 1 else residues[0]
    else:
        # Column is all gaps
        consensus += "-"
        degenerate_consensus += "-"

print("Consensus:", consensus)
print("Degenerate consensus:", degenerate_consensus)

# Create a DataFrame for sequence logo
df = pd.DataFrame(counts_matrix)
df.index.name = "Position"

# Plot logo
plt.figure(figsize=(15,4))
logo = logomaker.Logo(df, color_scheme='chemistry')
plt.title("Sequence Logo")

ticks=True
if ticks:
    # 6. Add Query_Sequence as tick labels
    query_sequence = None
    # Access records using the .sequences attribute for modern Bio.Align objects
    # Now reload using AlignIO (old API keeps dashes)
    msa = AlignIO.read("aligned_with_query.fasta", "fasta")

    # Extract the gappy sequence
    query_sequence = str([rec.seq for rec in msa if rec.id == "Query_Sequence"][0])
    for record in msa:
        if record.id == "Query_Sequence":
            query_sequence = str(record.seq)
            break
    print("Query Sequence:", query_sequence)

    if query_sequence:
        # Set tick positions to correspond to the positions in the sequence
        tick_positions = range(len(query_sequence))
        # Set tick labels to the characters of the query sequence
        plt.gca().set_xticks(tick_positions)
        plt.gca().set_xticklabels(list(query_sequence))
        # Optional: Rotate labels if they overlap
        plt.xticks(rotation=0)
        plt.tick_params(axis='x', which='major', labelsize=6) # Adjust label size if needed

plt.show()


## Report Questions

### Q9. Does your mutated sequence conform to the sequence logo? Note one or more agreements or substantial differences.

As an **optional** last step, let's have a look at the distribution of all these sequences, and the similarities of all of them compared to all others.

In [ ]:
#@title Analyze and visualize sequence identity distribution
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Find the query and original sequences in the alignment
query_sequence_id = "Query_Sequence"
original_sequence_id = "Original_Sequence"
query_seq = None
original_seq = None

for record in alignment:
    if record.id == query_sequence_id:
        query_seq = record
    elif record.id == original_sequence_id:
        original_seq = record

if query_seq is None or original_seq is None:
    print("Error: Query or Original sequence not found in the alignment.")
else:
    identity_data = []
    # Iterate through each sequence in the alignment
    for record in alignment:
        # Skip the query and original sequences themselves
        if record.id == query_sequence_id or record.id == original_sequence_id:
            continue

        # Calculate sequence identity against original sequence
        identical_residues_original = 0
        aligned_length_original = 0
        for i in range(alignment.get_alignment_length()):
            original_residue = original_seq[i]
            current_residue = record[i]

            if original_residue != '-' and current_residue != '-':
                aligned_length_original += 1
                if original_residue == current_residue:
                    identical_residues_original += 1

        identity_original = (identical_residues_original / aligned_length_original) * 100 if aligned_length_original > 0 else 0.0

        # Calculate sequence identity against query sequence
        identical_residues_query = 0
        aligned_length_query = 0
        for i in range(alignment.get_alignment_length()):
            query_residue = query_seq[i]
            current_residue = record[i]

            if query_residue != '-' and current_residue != '-':
                aligned_length_query += 1
                if query_residue == current_residue:
                    identical_residues_query += 1

        identity_query = (identical_residues_query / aligned_length_query) * 100 if aligned_length_query > 0 else 0.0

        identity_data.append({'ID': record.id, 'Identity_vs_Original': identity_original, 'Identity_vs_Query': identity_query})

    # Create a pandas DataFrame
    identity_df = pd.DataFrame(identity_data)

    # Plotting
    plt.figure(figsize=(12, 6))

    # Histogram of identity against original
    plt.subplot(1, 2, 1)
    sns.histplot(identity_df['Identity_vs_Original'], bins=20, kde=True)
    plt.title('Distribution of Sequence Identity vs. Original Sequence')
    plt.xlabel('Sequence Identity (%)')
    plt.ylabel('Frequency')

    # Scatterplot of identity vs original vs identity vs query
    plt.subplot(1, 2, 2)
    sns.scatterplot(data=identity_df, x='Identity_vs_Original', y='Identity_vs_Query')
    plt.title('Sequence Identity vs. Original vs. Query')
    plt.xlabel('Identity vs. Original (%)')
    plt.ylabel('Identity vs. Query (%)')

    plt.tight_layout()
    plt.show()

## Optional Report Question (advanced)

### Q10. (OPTIONAL) Reflect on the distribution of all pairwise distances among the 3000 sequences found in biologicaldatabases.
Suggestion: If you think of fold-space as points on a graph, how are those points connected? How can so many points be so far apart from each other within the same sequence space?